# 05b · Timestral-209M, zero-shot, full replay on the 2026-10-25 cut

**What this notebook does (one sentence):** it asks the pretrained Timestral-209M model to forecast the next
168 hours (one week) from every replay origin of every device/target pair, **without training it on our data**
("zero-shot"), and saves the forecasts in the same file layout as the Chronos-2 runs so notebook 06 can compare them.

**Role in the study:** PRIMARY size (declared before the run).
The twin notebook runs the 123M model with identical code; only `MODEL_ID` differs.

## Pre-declared choices (written BEFORE the run, never changed after seeing results)
| choice | value | why |
|---|---|---|
| model | `mistralai/Timestral-209M` | 209M is the primary |
| training | none (zero-shot) | decided 2026-11-24; fine-tuning only if zero-shot is near/above zero-shot Chronos-2 |
| context | 2048 h (about 85 days) | same as `chronos2_zeroshot_ctx2048` |
| horizon | 168 h | same as every model in the study |
| covariates | the same 8 as notebook 05 | fairness: Chronos-2 had them too |
| origins | local 00/06/12/18 from 2026-09-01, as many as have a full 168 h of actuals | same rule as notebook 05 |
| devices | all 13 devices, both targets (`grid_export` only where PV exists) | the frozen cohort |
| quantile output | 0.1, 0.25, 0.5, 0.75, 0.9; point forecast = median (0.5) | same as Chronos-2 |
| post-processing | clip at 0, sort quantiles; **no** recalibration | recalibration is a separate, later, exploratory step |

**Caveat that applies to every model in the study:** the weather covariates are the *observed* weather
(a "perfect forecast"), so all results are an **upper bound** on what is achievable with real weather forecasts.
Timestral's benchmark claims are self-reported by its authors, which is why we test it ourselves.

**Where outputs go:** forecasts are written to `runs/timestral/<RUN_NAME>/`, which is gitignored,
so nothing here changes tracked files.

## 1. Parameters
Everything you might want to change lives in this one cell. Change nothing after looking at results.
`DRY_WINDOWS` is for testing the notebook only: `None` means "do the full run".

In [ ]:
# This first cell is tagged `parameters` (papermill convention), like in notebook 05.
from pathlib import Path

MODEL_ID = "mistralai/Timestral-209M"   # the pretrained model downloaded from Hugging Face
RUN_NAME = "timestral-209M-zeroshot-cov"  # name of the output folder and of the model in the output files

HORIZON = 168            # hours to forecast from each origin (1 week)
CONTEXT = 2048           # hours of history the model may look at before each origin
MIN_CONTEXT_OBS = 24     # need at least one day of real (non-NaN) history, same rule as notebook 05
LOCAL_TZ = "Europe/Rome" # calendar features and origin times use local time
CUTOFF = "2026-09-01"    # replay starts here (local date); everything before is "training period" for other models
REPLAY_STRIDE_H = 6      # one origin every 6 hours: local 00, 06, 12, 18
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]   # the levels the study scores
WINDOWS_PER_BATCH = 5    # forecasts per GPU forward pass (small on purpose: the RTX 3050 has only 4 GB)
DRY_WINDOWS = None   # None = full run. A number = only forecast that many windows (testing)

# Paths. The data is the frozen cut, read-only. Outputs go to runs/timestral/, which is gitignored.
REPO = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "pyproject.toml").exists())
CUT_DATA = REPO / "runs/cut-2026-10-25/data"
CHRONOS_RUNS = REPO / "runs/gx10/nb05"                  # existing Chronos-2 forecasts, used only for comparison
OUT_DIR = REPO / "runs/timestral" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "parts").mkdir(exist_ok=True)               # one checkpoint file per (device, target) pair

## 2. Environment and hardware
**What happens here:** we import the libraries and pick the compute device. A **GPU** (graphics card) does the
huge number of small multiplications inside a neural network in parallel, so it is much faster than the **CPU**.
The laptop has an RTX 3050 with 4 GB of memory ("VRAM"); the code uses it automatically and falls back to the CPU
if no GPU is found. We print versions so the run can be reproduced later.

In [ ]:
import hashlib, json, time, warnings
import numpy as np
import pandas as pd
import torch
import timestral
from timestral import TimestralForecaster

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), "| VRAM %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print("timestral package:", Path(timestral.__file__).parent)
print("model:", MODEL_ID, "| output folder:", OUT_DIR)

## 3. Data integrity check (are we really on the 2026-10-25 cut?)
**What happens here:** every frozen file has a *SHA-256 hash* recorded in `SHA256SUMS`. A hash is a fingerprint:
a long number computed from the file's bytes. Change one byte and the fingerprint changes completely. If our
file's fingerprint equals the recorded one, the file is exactly the frozen data and the comparison with the
other models is valid. If not, we stop on purpose.

In [ ]:
def sha256(path):
    """Fingerprint of a file, read in 1 MB pieces so memory stays small."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Each SHA256SUMS line is "<hash>  <filename>"; build {filename: hash}.
recorded = dict(reversed(line.split()) for line in (CUT_DATA / "SHA256SUMS").read_text().splitlines() if line.strip())
for name in ["processed_hourly_2026-10-25.parquet", "device_quality_2026-10-25.csv"]:
    ok = sha256(CUT_DATA / name) == recorded[name]
    print("OK      " if ok else "MISMATCH", name)
    assert ok, f"{name} differs from the frozen cut: stop"

## 4. Load the data and build one continuous hourly series per device
**What happens here:** `processed_hourly` has one row per device per hour (UTC). For each device we make a
series with *no missing hours*: hours without a measurement become `NaN` ("not a number" = unknown).
Then "the 2048 hours before an origin" is just an array slice, with no date arithmetic to get wrong.

`HAS_PV` marks devices with solar panels. Only they can *export* energy to the grid, so
`grid_export` is forecast only for them (same as notebook 05).

In [ ]:
PT = pd.read_parquet(CUT_DATA / "processed_hourly_2026-10-25.parquet")
quality = pd.read_csv(CUT_DATA / "device_quality_2026-10-25.csv").set_index("device_id")
HAS_PV = quality["has_pv"].astype(bool).to_dict()

# {device: hourly DataFrame}. drop_duplicates keeps the last row if an hour appears twice;
# asfreq("h") inserts the missing hours as NaN rows.
DEV = {d: g.drop_duplicates("ts_hour", keep="last").set_index("ts_hour").sort_index().asfreq("h")
       for d, g in PT.groupby("device_id")}

ALL_DEVICES = sorted(DEV)
PAIRS = [(d, t) for t in ["grid_export", "grid_import"] for d in ALL_DEVICES if t != "grid_export" or HAS_PV.get(d, False)]
print(len(ALL_DEVICES), "devices ->", len(PAIRS), "device/target pairs")

## 5. Covariates: the same 8 as notebook 05
**What a covariate is:** an extra series that helps explain the target. Electricity use depends on the hour of day,
the weekday and holidays; solar export depends on sunshine. We give the model, for *every hour of the context
and of the forecast week*, these 8 numbers:

| covariate | idea |
|---|---|
| `hour_sin`, `hour_cos` | hour of day as a point on a circle |
| `dow_sin`, `dow_cos` | day of week as a point on a circle |
| `is_holiday` | 1 on Italian public holidays |
| `global_tilted_irradiance`, `temperature_2m`, `cloud_cover` | observed weather at the site |

**Math, why sin and cos?** Hour 23 and hour 0 are neighbours, but the numbers 23 and 0 are far apart. Mapping the
hour to an angle  `theta = 2*pi*hour/24`  and using  `(sin theta, cos theta)`  puts 23:00 and 00:00 next to each
other on a circle. Two numbers are needed because sin alone cannot tell 6:00 from 18:00.

**Caveat:** the weather is the *observed* weather at the forecast hour (a "perfect forecast"), so the results are an
upper bound. All models in the study share this caveat.

In [ ]:
# Italian public holidays (copied unchanged from notebook 05).
HOLIDAYS = {pd.Timestamp(d).date() for d in [
    "2025-01-01", "2025-01-06", "2025-04-21", "2025-04-25", "2025-05-01", "2025-06-02",
    "2025-08-15", "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-06", "2026-04-25", "2026-05-01", "2026-06-02",
    "2026-08-15", "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26"]}
WEATHER_COLS = ["global_tilted_irradiance", "temperature_2m", "cloud_cover"]

# ONE common hourly UTC grid ("HOURS"): starts CONTEXT hours before the first data point (so the first origin has a
# full context) and ends HORIZON hours after the last (so covariates exist for the whole forecast week).
HOURS = pd.date_range(PT["ts_hour"].min() - pd.Timedelta(hours=CONTEXT),
                      PT["ts_hour"].max() + pd.Timedelta(hours=HORIZON), freq="h")
local = HOURS.tz_convert(LOCAL_TZ)     # calendar features follow LOCAL clock time, not UTC

COV = pd.DataFrame({
    "hour_sin": np.sin(2 * np.pi * local.hour / 24),
    "hour_cos": np.cos(2 * np.pi * local.hour / 24),
    "dow_sin": np.sin(2 * np.pi * local.dayofweek / 7),
    "dow_cos": np.cos(2 * np.pi * local.dayofweek / 7),
    "is_holiday": pd.Index(local.date).isin(HOLIDAYS).astype(float)}, index=HOURS)
# One site => the weather is the same for every device: take the first row per hour, interpolate small gaps.
COV[WEATHER_COLS] = (PT.groupby("ts_hour")[WEATHER_COLS].first().reindex(HOURS).interpolate(limit_direction="both"))
COV_ARR = COV.to_numpy(np.float32)     # shape (n_hours, 8); float32 because the model works in 32-bit numbers

# Target series per (device, target) on the same grid; NaN where the meter had a gap.
Y = {(d, t): DEV[d][t].reindex(HOURS).to_numpy(np.float32) for d, t in PAIRS}

def pos(ts):
    """Row number of a UTC timestamp on the HOURS grid (whole hours since the grid start)."""
    return int((pd.Timestamp(ts) - HOURS[0]) // pd.Timedelta(hours=1))

print("covariates:", list(COV.columns), "| NaNs left in covariates:", int(np.isnan(COV_ARR).sum()))
assert not np.isnan(COV_ARR).any(), "covariates must be complete"

## 6. The forecast origins (same rule as notebook 05's replay)
**Vocabulary.** An **origin** is the moment a forecast is issued. A forecast from origin `o` has seen every
label *before* `o` and predicts hours `o, o+1, ..., o+167` (**lead** `h = 1..168`). No look-ahead: nothing at or
after `o` is ever shown to the model as target history.

**Which origins?** Every 6 hours (local 00:00, 06:00, 12:00, 18:00) starting 2026-09-01, up to the last origin
whose full 168-hour week still lies inside the data. We additionally skip an origin if its context has fewer than
24 real observations (the model would have almost nothing to read; notebook 05 uses the same rule).

In [ ]:
CUT = pd.Timestamp(CUTOFF, tz=LOCAL_TZ).tz_convert("UTC")
PATH_H = pd.Timedelta(hours=HORIZON)

JOBS = []   # one entry per forecast window: (device, target, origin)
for d, t in PAIRS:
    last_origin = DEV[d].index[-1] - PATH_H + pd.Timedelta(hours=1)    # the full week must exist in the data
    for o in pd.date_range(CUT, last_origin, freq=f"{REPLAY_STRIDE_H}h"):
        p = pos(o)
        ctx_obs = np.isfinite(Y[d, t][p - CONTEXT:p]).sum()            # real values in the context
        if ctx_obs >= MIN_CONTEXT_OBS:
            JOBS.append((d, t, o))
JOBS = pd.DataFrame(JOBS, columns=["device_id", "target", "origin"])
print(f"{len(JOBS)} forecast windows | {JOBS.origin.min()} -> {JOBS.origin.max()}")
print(JOBS.groupby("target").size().to_string())
if DRY_WINDOWS:   # testing only: keep a few windows spread across the pairs
    JOBS = JOBS.groupby(["device_id", "target"]).head(1).head(DRY_WINDOWS)
    print("DRY RUN: only", len(JOBS), "windows")

## 7. Building the input of one forecast
For one (device, target, origin) the model receives
- **context** `y[o-2048 .. o-1]`: the last 2048 hourly values of the target (NaN allowed where the meter had a gap);
- **known covariates** over `o-2048 .. o+167`: the 8 series above for the context *and* the forecast week.

**Math, patches.** Timestral does not read single hours. It cuts the series into **patches** of 32 consecutive
values and treats each patch like a "word" for a transformer (the same architecture family as language models).
2048 hours / 32 = **64 patches** of context. Fewer, bigger tokens make long contexts cheap. The model then outputs,
for every future hour, **21 quantiles** (0.01, 0.05, ..., 0.99) instead of one number.

In [ ]:
def window(device, target, origin):
    """One forecast input: context (T,) and covariates (8, T+H), both float32."""
    p = pos(origin)
    assert HOURS[p] == origin and p >= CONTEXT          # the origin really is on the grid, with enough history
    y = Y[device, target][p - CONTEXT:p]                # labels strictly BEFORE the origin (no leakage)
    cov = COV_ARR[p - CONTEXT:p + HORIZON].T            # (8, 2048+168): one row per covariate
    return y, np.ascontiguousarray(cov)

## 8. The forecast loop (the slow part) with crash-safe checkpoints
**What happens here:** for each device/target pair we forecast all its origins, in batches of
`WINDOWS_PER_BATCH` windows per GPU pass, and write the result to `parts/<device>__<target>.parquet`
**immediately**. If the run is interrupted (power, crash), re-running the notebook skips pairs that are already
saved and continues, so hours of work are never lost.

**Post-processing of the model output (identical to Chronos-2 in notebook 05):**
1. keep only our 5 quantile levels out of the model's 21;
2. **clip at 0**: grid import/export energy cannot be negative, `q <- max(0, q)`;
3. **sort across quantiles** so they can never cross, e.g. the 0.9 quantile can never be below the 0.5 one.
   Clipping and sorting only ever *fix* impossible values; they do not use any actuals.

The **point forecast** is the median (quantile 0.5). The median is the value that actuals fall below 50% of
the time; it is the best single number when the score is the absolute error (MAE), explained below.

In [ ]:
def level_index(levels):
    """Position of each wanted quantile level inside the model's 21 output levels."""
    lv = np.asarray(levels, dtype=float)
    return [int(np.argmin(np.abs(lv - q))) for q in QUANTILES]

forecaster = TimestralForecaster.from_pretrained(MODEL_ID, device=DEVICE)   # downloaded once, then cached on disk
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

def forecast_pair(device, target, origins):
    """Forecast every origin of one pair; returns a long DataFrame (one row per origin x lead)."""
    out = []
    for i in range(0, len(origins), WINDOWS_PER_BATCH):
        batch = origins[i:i + WINDOWS_PER_BATCH]
        ctx, cov = zip(*(window(device, target, o) for o in batch))
        context = np.stack(ctx)[:, None, :]                      # (batch, 1 variate, 2048)
        known = np.stack(cov)                                    # (batch, 8, 2048+168)
        fc = forecaster.forecast(context, prediction_length=HORIZON, known_covariates=known)
        q = fc.quantiles.float().cpu().numpy() if torch.is_tensor(fc.quantiles) else np.asarray(fc.quantiles)
        q = q[:, 0][:, :, level_index(fc.quantile_levels)]       # (batch, 168, 5)
        q = np.sort(np.maximum(0.0, q), axis=-1)                 # clip at 0, then enforce non-crossing
        for o, qq in zip(batch, q):
            out.append(pd.DataFrame({"origin": o, "horizon": np.arange(1, HORIZON + 1),
                                     **{f"q{lv}": qq[:, k] for k, lv in enumerate(QUANTILES)}}))
    return pd.concat(out, ignore_index=True)

t_start, done_windows = time.perf_counter(), 0
for n, ((d, t), g) in enumerate(JOBS.groupby(["device_id", "target"]), 1):
    part = OUT_DIR / "parts" / f"{d}__{t}.parquet"
    if part.exists():                                            # resume support
        print(f"[{n}] {d} {t}: already done, skipping")
        continue
    t0 = time.perf_counter()
    df = forecast_pair(d, t, list(g["origin"]))
    df.insert(0, "target", t); df.insert(0, "device_id", d)
    df.to_parquet(part, index=False)                             # checkpoint right away
    done_windows += len(g)
    print(f"[{n}/{JOBS.groupby(['device_id', 'target']).ngroups}] {d} {t}: {len(g)} windows in {time.perf_counter() - t0:.0f}s"
          f" | total {(time.perf_counter() - t_start) / 60:.1f} min")
SEC_PER_WINDOW = (time.perf_counter() - t_start) / max(done_windows, 1)
PEAK_VRAM_GB = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else None
print(f"done: {done_windows} new windows, {SEC_PER_WINDOW:.2f} s/window, peak VRAM {PEAK_VRAM_GB} GB")

## 9. Assemble the result in the same layout as the Chronos-2 files
**What happens here:** we glue the per-pair checkpoints together and add the true value (`actual`) and the
timestamp of the target hour (`ts_hour = origin + (lead-1) hours`). The columns match `replay.parquet` of
notebook 05 (`model, scope, device_id, target, origin, horizon, ts_hour, actual, point, q0.1 ... q0.9`), so
notebook 06 can read it exactly like the other models. Rows with no actual (a meter gap in the target week)
are dropped, as in notebook 05.

In [ ]:
fc = pd.concat([pd.read_parquet(f) for f in sorted((OUT_DIR / "parts").glob("*.parquet"))], ignore_index=True)
fc = fc.merge(JOBS, on=["device_id", "target", "origin"])        # keep only the windows of THIS run (matters for dry runs)
fc["ts_hour"] = fc["origin"] + pd.to_timedelta(fc["horizon"] - 1, unit="h")
# actual value of every (pair, target hour), looked up on the hourly grid
fc["actual"] = [Y[d, t][pos(ts)] for d, t, ts in zip(fc["device_id"], fc["target"], fc["ts_hour"])]
fc["point"] = fc["q0.5"]                                          # point forecast = median
fc["model"], fc["scope"] = RUN_NAME, "pooled"                     # "pooled": one model for all devices, as for Chronos-2
n_before = len(fc)
fc = fc.dropna(subset=["actual"])                                 # score only hours that have an actual
COLS = ["model", "scope", "device_id", "target", "origin", "horizon", "ts_hour", "actual", "point"] + [f"q{l}" for l in QUANTILES]
fc = fc[COLS].astype({**dict.fromkeys(["model", "scope", "device_id", "target"], object),
                      **dict.fromkeys(["point", "actual"] + [f"q{l}" for l in QUANTILES], "float64")})
fc.to_parquet(OUT_DIR / "replay.parquet", index=False)
_info = OUT_DIR / "run_info.json"
if done_windows == 0 and _info.exists():      # resumed run that forecast nothing new: keep the real timing
    _old = json.load(open(_info)); SEC_PER_WINDOW, PEAK_VRAM_GB = _old["sec_per_window"], _old["peak_vram_gb"]
json.dump({"model_id": MODEL_ID, "device": DEVICE, "context": CONTEXT, "horizon": HORIZON, "windows": len(JOBS),
           "rows": len(fc), "rows_dropped_no_actual": n_before - len(fc), "sec_per_window": SEC_PER_WINDOW,
           "peak_vram_gb": PEAK_VRAM_GB, "torch": torch.__version__},
          open(OUT_DIR / "run_info.json", "w"), indent=1)
print(f"{len(fc):,} rows saved to {OUT_DIR / 'replay.parquet'} ({n_before - len(fc):,} dropped: no actual)")

## 10. Sanity checks (declared rules, same spirit as the smoke test)
- **C1 finite:** no NaN / infinity in any forecast.
- **C2 no crossing:** `q0.1 <= q0.25 <= q0.5 <= q0.75 <= q0.9` on every row.
- **C3 non-negative:** no forecast below 0.
- **C4 complete:** every planned window produced a full 168-lead forecast.

In [ ]:
Q = fc[[f"q{l}" for l in QUANTILES]].to_numpy()
checks = {"C1 finite": bool(np.isfinite(Q).all()),
          "C2 no crossing": bool((np.diff(Q, axis=1) >= -1e-9).all()),
          "C3 non-negative": bool((Q >= 0).all()),
          "C4 complete (windows x 168 rows >= rows kept)": bool(len(fc) <= len(JOBS) * HORIZON and
                                                                 fc.groupby(["device_id", "target", "origin"]).ngroups <= len(JOBS))}
for k, v in checks.items():
    print("PASS" if v else "FAIL", k)
assert all(checks.values()), "a sanity check failed: do not use these forecasts"

## 11. First look: how does it score next to Chronos-2? (descriptive, not the final test)
**This is only a quick look.** The proper comparison (all models, Holm-corrected Diebold-Mariano tests) happens in
notebook 06. Here we score on the rows that exist for **both** Timestral and Chronos-2, so the comparison is fair.

**The three scores, with the math (`y` = actual, `f` = forecast, `n` = number of rows):**
- **MAE** (mean absolute error) = `(1/n) * sum |y - f|`. In kWh per hour; lower is better. With the median as the
  forecast this is the natural score: the median is exactly the number that minimises the expected absolute error.
- **Pinball loss** for quantile level `tau`: `max(tau*(y-q), (tau-1)*(y-q))`, where `q` is the forecast quantile.
  If the actual is above the quantile (`y > q`) we pay `tau` per unit of miss; if below, we pay `1-tau` per unit.
  For `tau = 0.9` being too low costs 9x more than being too high, which pushes `q` to the value that 90% of actuals
  fall under. We average over the five levels; lower is better. It scores the *whole* uncertainty band, not just the median.
- **cov80** = share of actuals that fall between the 0.1 and 0.9 quantiles. A well-calibrated 80% band gives about
  0.80. Much higher = bands too wide (cautious); much lower = overconfident.

In [ ]:
def pinball(y, q, level):
    """Quantile loss for one level; asymmetric so each quantile is pushed to its own level."""
    d = y - q
    return np.maximum(level * d, (level - 1) * d)

KEYS = ["device_id", "target", "origin", "horizon"]
refs = {
    "chronos2_zeroshot_ctx2048": ("cut1025-chronos2-zeroshot-ctx2048-h168", "chronos2_zeroshot_ctx2048"),
    "chronos2 (fine-tuned)":     ("cut1025-chronos2-h168", "chronos2"),
}
frames = {RUN_NAME: fc}
for label, (run, model) in refs.items():
    r = pd.read_parquet(CHRONOS_RUNS / run / "replay.parquet")
    frames[label] = r[r["model"] == model][COLS]

# Keep only rows that exist in EVERY model, so all models are scored on exactly the same hours.
common = None
for df in frames.values():
    idx = pd.MultiIndex.from_frame(df[KEYS])
    common = idx if common is None else common.intersection(idx)
print("rows scored by all models:", len(common))

def summarise(df):
    m = df.set_index(KEYS).loc[common].reset_index()
    y = m["actual"]
    return pd.Series({"MAE": (y - m["point"]).abs().mean(),
                      "pinball": np.mean([pinball(y, m[f"q{l}"], l).mean() for l in QUANTILES]),
                      "cov80": ((y >= m["q0.1"]) & (y <= m["q0.9"])).mean()})

summary = pd.DataFrame({name: summarise(df) for name, df in frames.items()}).T
print("\nAll pairs pooled (lower MAE and pinball are better; cov80 should be near 0.80):")
display(summary.round(4))

per_target = {}
for name, df in frames.items():
    m = df.set_index(KEYS).loc[common].reset_index()
    per_target[name] = {t: (g["actual"] - g["point"]).abs().mean() for t, g in m.groupby("target")}
print("MAE by target:")
display(pd.DataFrame(per_target).T.round(4))
summary.to_csv(OUT_DIR / "first_look_summary.csv")

## 12. Plots
Five figures, each saved as a PNG in the output folder. The Chronos-2 lines come from the existing runs and use
the same common rows as the table above, so every line is the average over exactly the same hours.

In [ ]:
import matplotlib.pyplot as plt
MODELS = {RUN_NAME: "#c0392b", "chronos2_zeroshot_ctx2048": "#5b2c6f", "chronos2 (fine-tuned)": "#2a9d8f"}
CM = {n: df.set_index(KEYS).loc[common].reset_index() for n, df in frames.items()}   # common rows only
for n, m in CM.items():
    m["ae"] = (m["point"] - m["actual"]).abs()

### 12.1 Accuracy against lead time
**What it shows:** the MAE for each lead `h = 1..168` (how many hours ahead the forecast looks), one line per
model, one panel per target. **Math:** `MAE(h) = mean over origins and devices of |y(o+h-1) - f(o,h)|`.
A line that rises slowly with `h` means the model keeps its skill further into the week. The bold line is a 6 h rolling
mean, which removes the daily wiggle caused by the hour of day the lead lands on.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharex=True)
for ax, target in zip(axes, ["grid_export", "grid_import"]):
    for n, col in MODELS.items():
        curve = CM[n][CM[n]["target"] == target].groupby("horizon")["ae"].mean()
        ax.plot(curve.index, curve.rolling(6, center=True, min_periods=1).mean(), color=col, lw=2, label=n)
    ax.set(title=f"{target}: MAE by lead (6 h rolling mean)", xlabel="lead (hours ahead)", ylabel="MAE (kWh/h)")
    ax.grid(alpha=.3)
axes[0].legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "plot_lead_mae.png", dpi=120); plt.show()

### 12.2 MAE per device
**What it shows:** the MAE of each device (all leads pooled), for the three models, so we can see whether a
gain is spread over all devices or comes from one or two. Devices differ a lot in size, so only the *relative*
position of the bars within one device is meaningful.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, target in zip(axes, ["grid_export", "grid_import"]):
    tab = pd.DataFrame({n: CM[n][CM[n]["target"] == target].groupby("device_id")["ae"].mean() for n in MODELS})
    w = 0.8 / len(MODELS)
    for i, (n, col) in enumerate(MODELS.items()):
        ax.bar(np.arange(len(tab)) + (i - 1) * w, tab[n], w, color=col, label=n)
    ax.set_xticks(range(len(tab)), [d[-5:] for d in tab.index], rotation=60, fontsize=8)
    ax.set(title=f"{target}: MAE per device", ylabel="MAE (kWh/h)"); ax.grid(alpha=.3, axis="y")
axes[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "plot_device_mae.png", dpi=120); plt.show()

### 12.3 Are the quantiles honest? (reliability)
**What it shows:** for each quantile level `tau`, the share of actuals that fell at or below the forecast quantile.
**Math:** if the quantile `q_tau` is right, `P(y <= q_tau) = tau`, so the points should lie on the diagonal.
Points *above* the diagonal at the high levels and *below* it at the low levels mean the band is too wide
(over-cautious); the opposite means over-confident. Many hours are exactly 0 (night for export), so the low
quantiles sit at 0 and "at or below" is already high there; read the 0.75 and 0.9 points first.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
for n, col in MODELS.items():
    m = CM[n]
    ax.plot(QUANTILES, [(m["actual"] <= m[f"q{l}"] + 1e-9).mean() for l in QUANTILES], "o-", color=col, label=n)
ax.set(xlabel="quantile level tau", ylabel="share of actuals at or below the quantile", title="Reliability (all pairs)")
ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "plot_reliability.png", dpi=120); plt.show()

### 12.4 One week, forecast against actual
**What it shows:** one origin on the PV device with the most export, the actual curve (black), Timestral's median
(red line) and its 10-90% band (shaded), and the Chronos-2 fine-tuned median (green) for comparison. This is the
only plot that lets you *see* what the numbers mean: does the model get the daily solar bell right, and does the band
contain the actuals?

In [ ]:
ex = fc[fc["target"] == "grid_export"]
dev = ex.groupby("device_id")["actual"].mean().idxmax()
origin = sorted(ex["origin"].unique())[len(ex["origin"].unique()) // 2]       # a mid-replay origin
a = fc[(fc.device_id == dev) & (fc.target == "grid_export") & (fc.origin == origin)].sort_values("horizon")
b = frames["chronos2 (fine-tuned)"]
b = b[(b.device_id == dev) & (b.target == "grid_export") & (b.origin == origin)].sort_values("horizon")
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(a["horizon"], a["q0.1"], a["q0.9"], color="#c0392b", alpha=.2, label="Timestral 10-90% band")
ax.plot(a["horizon"], a["point"], color="#c0392b", lw=1.8, label="Timestral median")
ax.plot(b["horizon"], b["point"], color="#2a9d8f", lw=1.4, label="Chronos-2 fine-tuned median")
ax.plot(a["horizon"], a["actual"], color="black", lw=1.4, label="actual")
ax.set(title=f"{dev}, grid_export, origin {origin:%Y-%m-%d %H:%M} UTC", xlabel="lead (hours)", ylabel="kWh/h")
ax.grid(alpha=.3); ax.legend(ncol=4, fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "plot_example_week.png", dpi=120); plt.show()

### 12.5 Forecast against actual (scatter) and error spread
**Left, scatter:** every hour as one point (forecast median on the x axis, actual on y). A perfect model lies on the
diagonal. **Right, error histogram:** `e = forecast - actual`; a narrow peak at 0 is good, and a skew to the right means
the model over-forecasts on average (positive bias). Both use all pairs of the target `grid_import` and
`grid_export` together, sampled to 30,000 hours for speed.

In [ ]:
m = CM[RUN_NAME].sample(min(30000, len(CM[RUN_NAME])), random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(m["point"], m["actual"], s=3, alpha=.15, color="#c0392b")
top = float(max(m["point"].max(), m["actual"].max()))
axes[0].plot([0, top], [0, top], "k--", lw=1)
axes[0].set(xlabel="forecast median", ylabel="actual", title=f"{RUN_NAME}: forecast vs actual"); axes[0].grid(alpha=.3)
e = (CM[RUN_NAME]["point"] - CM[RUN_NAME]["actual"])
axes[1].hist(e.clip(-1.5, 1.5), bins=120, color="#c0392b")
axes[1].axvline(0, color="k", lw=1); axes[1].axvline(e.mean(), color="blue", lw=1, ls="--", label=f"bias {e.mean():+.3f}")
axes[1].set(xlabel="forecast - actual (kWh/h, clipped at +-1.5)", title="Error distribution"); axes[1].legend()
fig.tight_layout(); fig.savefig(OUT_DIR / "plot_scatter_errors.png", dpi=120); plt.show()

## 13. How to read this and what comes next
- If Timestral-209M is clearly better or worse than `chronos2_zeroshot_ctx2048`, notebook 06 will say whether the
  difference is **statistically real**. The **Diebold-Mariano (DM) test** looks at the per-hour loss difference
  `d_t = |y_t - f_A,t| - |y_t - f_B,t|` and asks whether its mean is different from zero, after accounting for the
  fact that forecast errors in neighbouring hours are correlated. Because we run many such tests, p-values are
  adjusted with the **Holm correction** (strictest p-value is multiplied by the number of tests, the next by one fewer, ...),
  which keeps the chance of any false alarm at 5%.
- A raw difference of a few percent between two models is **not** evidence by itself.
- Remember the upper-bound caveat: observed weather was used for every model.
- The 123M/209M pair shows whether the extra parameters pay off on this data.